In [1]:
import numpy as np
import pandas as pd
import datetime

# date → has date only (orders dataset)
# time → has date + time (messages dataset)
date = pd.read_csv('orders.csv')
time = pd.read_csv('messages.csv')

print("Orders dataset:")
print(date.head())
print("\nDate column type:", date['date'].dtype)  # object → string not datetime

print("\nMessages dataset:")
print(time.head())
print("\nDate column type:", time['date'].dtype)  # object → string not datetime

Orders dataset:
         date  product_id  city_id  orders
0  2019-12-10        5628       25       3
1  2018-08-15        3646       14     157
2  2018-10-23        1859       25       1
3  2019-08-17        7292       25       1
4  2019-01-06        4344       25       3

Date column type: str

Messages dataset:
                  date                                                msg
0  2013-12-15 00:50:00                           ищу на сегодня мужика 37
1  2014-04-29 23:40:00   ПАРЕНЬ БИ ИЩЕТ ДРУГА СЕЙЧАС!! СМС ММС 0955532826
2  2012-12-30 00:21:00           Днепр.м 43 позн.с д/ж *.о  067.16.34.576
3  2014-11-28 00:31:00  КИЕВ ИЩУ Д/Ж ДО 45 МНЕ СЕЙЧАС СКУЧНО 093 629 9...
4  2013-10-26 23:11:00    Зая я тебя никогда не обижу  люблю тебя!)  Даше

Date column type: str


In [2]:
# pd.to_datetime() → converts string to proper datetime type
# Without this → you can't use .dt.year, .dt.month etc.

date['date'] = pd.to_datetime(date['date'])
time['date'] = pd.to_datetime(time['date'])

print("After conversion:")
print("Orders date type  :", date['date'].dtype)   # datetime64[ns]
print("Messages date type:", time['date'].dtype)   # datetime64[ns]

print("\nSample after conversion:")
print(date['date'].head(3))

After conversion:
Orders date type  : datetime64[us]
Messages date type: datetime64[us]

Sample after conversion:
0   2019-12-10
1   2018-08-15
2   2018-10-23
Name: date, dtype: datetime64[us]


In [3]:
# .dt.year → extracts the year part
date['date_year'] = date['date'].dt.year

print(date[['date','date_year']].head(5))

        date  date_year
0 2019-12-10       2019
1 2018-08-15       2018
2 2018-10-23       2018
3 2019-08-17       2019
4 2019-01-06       2019


In [4]:
# .dt.month → month as number (1-12)
date['date_month_no'] = date['date'].dt.month

# .dt.month_name() → month as full name string
date['date_month_name'] = date['date'].dt.month_name()

print(date[['date','date_month_no','date_month_name']].head(5))

        date  date_month_no date_month_name
0 2019-12-10             12        December
1 2018-08-15              8          August
2 2018-10-23             10         October
3 2019-08-17              8          August
4 2019-01-06              1         January


In [5]:
# .dt.day → day of the month (1-31)
date['date_day'] = date['date'].dt.day

# .dt.dayofweek → day of week as number
# Monday=0, Tuesday=1, ... Sunday=6
date['date_dow'] = date['date'].dt.dayofweek

# .dt.day_name() → day name as string
date['date_dow_name'] = date['date'].dt.day_name()

print(date[['date','date_day','date_dow','date_dow_name']].head(5))

        date  date_day  date_dow date_dow_name
0 2019-12-10        10         1       Tuesday
1 2018-08-15        15         2     Wednesday
2 2018-10-23        23         1       Tuesday
3 2019-08-17        17         5      Saturday
4 2019-01-06         6         6        Sunday


In [6]:
# np.where(condition, value_if_true, value_if_false)
# condition → day name is Saturday or Sunday
# true  → 1 (is weekend)
# false → 0 (is weekday)

date['date_is_weekend'] = np.where(
    date['date_dow_name'].isin(['Saturday', 'Sunday']),
    1,   # weekend → 1
    0    # weekday → 0
)

print(date[['date','date_dow_name','date_is_weekend']].head(5))

print("\nWeekend distribution:")
print(date['date_is_weekend'].value_counts())

        date date_dow_name  date_is_weekend
0 2019-12-10       Tuesday                0
1 2018-08-15     Wednesday                0
2 2018-10-23       Tuesday                0
3 2019-08-17      Saturday                1
4 2019-01-06        Sunday                1

Weekend distribution:
date_is_weekend
0    716
1    284
Name: count, dtype: int64


In [7]:
# .dt.isocalendar().week → ISO week number (1-53)
# older pandas used .dt.week (deprecated)

date['date_week'] = date['date'].dt.isocalendar().week.astype(int)

print(date[['date','date_month_no','date_week']].head(5))

        date  date_month_no  date_week
0 2019-12-10             12         50
1 2018-08-15              8         33
2 2018-10-23             10         43
3 2019-08-17              8         33
4 2019-01-06              1          1


In [8]:
# .dt.quarter → Q1(Jan-Mar)=1, Q2(Apr-Jun)=2,
#               Q3(Jul-Sep)=3, Q4(Oct-Dec)=4

date['quarter'] = date['date'].dt.quarter

print(date[['date','date_month_no','quarter']].head(5))

print("\nQuarter distribution:")
print(date['quarter'].value_counts().sort_index())

        date  date_month_no  quarter
0 2019-12-10             12        4
1 2018-08-15              8        3
2 2018-10-23             10        4
3 2019-08-17              8        3
4 2019-01-06              1        1

Quarter distribution:
quarter
1    149
2    162
3    294
4    395
Name: count, dtype: int64


In [9]:
# Semester = half of the year
# Q1 + Q2 → Semester 1 (Jan-Jun)
# Q3 + Q4 → Semester 2 (Jul-Dec)

date['semester'] = np.where(
    date['quarter'].isin([1, 2]),
    1,   # first half of year
    2    # second half of year
)

print(date[['date','quarter','semester']].head(5))

        date  quarter  semester
0 2019-12-10        4         2
1 2018-08-15        3         2
2 2018-10-23        4         2
3 2019-08-17        3         2
4 2019-01-06        1         1


In [10]:
# datetime.datetime.today() → current date and time right now
today = datetime.datetime.today()
print("Today:", today)

# subtracting date from today → gives timedelta (duration)
elapsed = today - date['date']
print("\nRaw timedelta:")
print(elapsed.head(3))

Today: 2026-07-11 16:00:24.028575

Raw timedelta:
0   2405 days 16:00:24.028575
1   2887 days 16:00:24.028575
2   2818 days 16:00:24.028575
Name: date, dtype: timedelta64[us]


In [12]:
today = datetime.datetime.today()

# days since
date['days_since'] = (today - date['date']).dt.days

# months since → divide days by 30.44 (average days in a month)
date['months_since'] = np.round(date['days_since'] / 30.44, 0)

# years since → divide days by 365.25 (average days in a year)
date['years_since'] = np.round(date['days_since'] / 365.25, 1)

print(date[['date','days_since','months_since','years_since']].head(5))


        date  days_since  months_since  years_since
0 2019-12-10        2405          79.0          6.6
1 2018-08-15        2887          95.0          7.9
2 2018-10-23        2818          93.0          7.7
3 2019-08-17        2520          83.0          6.9
4 2019-01-06        2743          90.0          7.5


In [13]:
# .dt.hour   → 0 to 23
# .dt.minute → 0 to 59
# .dt.second → 0 to 59

time['hour'] = time['date'].dt.hour
time['min']  = time['date'].dt.minute
time['sec']  = time['date'].dt.second

print(time[['date','hour','min','sec']].head(5))

                 date  hour  min  sec
0 2013-12-15 00:50:00     0   50    0
1 2014-04-29 23:40:00    23   40    0
2 2012-12-30 00:21:00     0   21    0
3 2014-11-28 00:31:00     0   31    0
4 2013-10-26 23:11:00    23   11    0


In [14]:
# .dt.time → extracts only the time portion (HH:MM:SS)
time['time_only'] = time['date'].dt.time

print(time[['date','time_only']].head(5))

                 date time_only
0 2013-12-15 00:50:00  00:50:00
1 2014-04-29 23:40:00  23:40:00
2 2012-12-30 00:21:00  00:21:00
3 2014-11-28 00:31:00  00:31:00
4 2013-10-26 23:11:00  23:11:00


In [15]:
# create meaningful time buckets from hour
# this is more useful than raw hour for some models

def time_of_day(hour):
    if   0 <= hour < 6:   return 'Night'
    elif 6 <= hour < 12:  return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                 return 'Night'

time['time_of_day'] = time['hour'].apply(time_of_day)

print(time[['date','hour','time_of_day']].head(10))
print("\nTime of day distribution:")
print(time['time_of_day'].value_counts())

                 date  hour time_of_day
0 2013-12-15 00:50:00     0       Night
1 2014-04-29 23:40:00    23       Night
2 2012-12-30 00:21:00     0       Night
3 2014-11-28 00:31:00     0       Night
4 2013-10-26 23:11:00    23       Night
5 2016-03-08 22:52:00    22       Night
6 2014-02-18 00:23:00     0       Night
7 2012-11-23 01:10:00     1       Night
8 2014-12-23 01:20:00     1       Night
9 2012-11-03 23:46:00    23       Night

Time of day distribution:
time_of_day
Night        998
Afternoon      2
Name: count, dtype: int64


In [16]:
today = datetime.datetime.today()

# in days
time['elapsed_days'] = (today - time['date']).dt.days

# in hours
time['elapsed_hours'] = np.round(
    (today - time['date']) / np.timedelta64(1, 'h'),
    1
)

# in minutes
time['elapsed_minutes'] = np.round(
    (today - time['date']) / np.timedelta64(1, 'm'),
    0
)

# in seconds
time['elapsed_seconds'] = np.round(
    (today - time['date']) / np.timedelta64(1, 's'),
    0
)

print(time[['date','elapsed_days','elapsed_hours',
            'elapsed_minutes','elapsed_seconds']].head(5))

                 date  elapsed_days  elapsed_hours  elapsed_minutes  \
0 2013-12-15 00:50:00          4591       110199.3        6611956.0   
1 2014-04-29 23:40:00          4455       106936.4        6416186.0   
2 2012-12-30 00:21:00          4941       118599.7        7115985.0   
3 2014-11-28 00:31:00          4243       101847.6        6110855.0   
4 2013-10-26 23:11:00          4640       111376.9        6682615.0   

   elapsed_seconds  
0      396717346.0  
1      384971146.0  
2      426959086.0  
3      366651286.0  
4      400956886.0  


In [17]:
# is it the first day of the month?
date['is_month_start'] = date['date'].dt.is_month_start.astype(int)

# is it the last day of the month?
date['is_month_end']   = date['date'].dt.is_month_end.astype(int)

print(date[['date','is_month_start','is_month_end']].head(5))

        date  is_month_start  is_month_end
0 2019-12-10               0             0
1 2018-08-15               0             0
2 2018-10-23               0             0
3 2019-08-17               0             0
4 2019-01-06               0             0


In [18]:
date['is_year_start'] = date['date'].dt.is_year_start.astype(int)
date['is_year_end']   = date['date'].dt.is_year_end.astype(int)

print(date[['date','is_year_start','is_year_end']].head(5))

        date  is_year_start  is_year_end
0 2019-12-10              0            0
1 2018-08-15              0            0
2 2018-10-23              0            0
3 2019-08-17              0            0
4 2019-01-06              0            0
